# Trabajo Final — Pipeline de pedidos de TiendaBSG
**Tópicos Avanzados para Data Science en Python**

**Alumno(a):**Fernando Cisneros Maciel
**Repositorio público:** https://github.com/fernandocisneros-cmyk/trabajo-final-spark

---

Eres el *data engineer* de **TiendaBSG**, una tienda online con despachos en 6 ciudades del Perú. Cada día el sistema de ventas deja un archivo CSV con los pedidos del día en la carpeta `datos/raw/`. Tu trabajo es procesar los pedidos de **octubre 2026** de principio a fin:

| Parte | Qué harás | Capítulo | Puntos |
|---|---|---|---|
| 1. Revisar los archivos | Explorar `raw/`, separar archivos vacíos y contar registros | 1. Línea de comandos | 3.5 |
| 2. Analizar con Spark | Limpiar, filtrar, agrupar y contar con DataFrames o Spark SQL | 3. Spark | 12 |
| 3. Cerrar el proceso | Mover los CSV procesados y dejar el resultado final listo | 1. Línea de comandos | 2 |
| 4. Entregar | Subir este notebook y el resultado a tu repositorio público | 2. Git | 2.5 |
| | | **Total** | **20** |

**Reglas**
- Ejecuta las celdas **en orden**. Si algo se rompe, vuelve a ejecutar la **Preparación** (reinicia los datos) y continúa.
- En la Parte 2 usa solo **DataFrames** (API de PySpark o Spark SQL). **No uses RDDs.**
- Responde las preguntas como **comentarios** (`# ...`) en tu propia celda de código.
- Antes de descargar el notebook: **Entorno de ejecución → Reiniciar y ejecutar todo**, para que se vean todas las salidas en GitHub.

## Preparación (no se califica)
Ejecuta estas tres celdas. Instalan PySpark, crean la sesión de Spark y generan los archivos del mes:

```
/content/trabajo_final/
├── datos/
│   ├── raw/          ← 31 archivos pedidos_2026-10-DD.csv (uno por día)
│   ├── procesados/   ← vacía: aquí moverás los CSV ya procesados (Parte 3)
│   └── cuarentena/   ← vacía: aquí moverás los archivos con problemas (Parte 1)
└── resultados/       ← vacía: aquí guardarás el resumen final (Partes 2 y 3)
```

In [1]:
# Java + PySpark (tarda ~1 minuto la primera vez)
import shutil
if shutil.which("java") is None:
    !apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q uninstall -y dataproc-spark-connect  # paquete de Colab que pide PySpark 4; no lo usamos
!pip -q install "pyspark==3.5.*"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("BSG-TrabajoFinal")
         .config("spark.ui.showConsoleProgress", "false")   # sin barras de progreso
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                     # oculta los logs de la JVM
print("Versión de Spark:", spark.version)

Versión de Spark: 3.5.9


In [3]:
# Genera los pedidos de octubre 2026 (volver a ejecutar = reiniciar los datos)
import csv, os, random
from datetime import date

TF = "/content/trabajo_final"
RAW = f"{TF}/datos/raw"
PROCESADOS = f"{TF}/datos/procesados"
CUARENTENA = f"{TF}/datos/cuarentena"
RESULTADOS = f"{TF}/resultados"

shutil.rmtree(TF, ignore_errors=True)
for carpeta in (RAW, PROCESADOS, CUARENTENA, RESULTADOS):
    os.makedirs(carpeta)

rng = random.Random(2026)
CIUDADES = {"Lima": 40, "Arequipa": 15, "Trujillo": 12, "Chiclayo": 10, "Cusco": 13, "Piura": 10}
CATALOGO = {  # producto: (categoria, precio de lista, peso en las ventas)
    "Laptop": ("Tecnologia", 3200.0, 8), "Celular": ("Tecnologia", 1500.0, 15),
    "Audifonos": ("Tecnologia", 180.0, 20), "Licuadora": ("Hogar", 250.0, 12),
    "Cafetera": ("Hogar", 320.0, 8), "Zapatillas": ("Moda", 280.0, 22),
    "Casaca": ("Moda", 220.0, 15)}
DEVOLUCION = {"Tecnologia": 4, "Hogar": 3, "Moda": 12}  # % de pedidos devueltos
PAGOS = {"Tarjeta": 40, "Yape": 30, "Efectivo": 15, "Transferencia": 15}
DIAS_VACIOS = {12, 25}  # el sistema falló y dejó estos archivos vacíos

id_pedido, dia_anterior = 0, []
for dia in range(1, 32):
    fecha = date(2026, 10, dia).isoformat()
    ruta = f"{RAW}/pedidos_{fecha}.csv"
    if dia in DIAS_VACIOS:
        open(ruta, "w").close()
        continue
    filas = []
    for _ in range(rng.randint(1500, 2500)):
        id_pedido += 1
        ciudad = rng.choices(list(CIUDADES), weights=CIUDADES.values())[0]
        producto = rng.choices(list(CATALOGO), weights=[v[2] for v in CATALOGO.values()])[0]
        categoria, precio_lista, _ = CATALOGO[producto]
        precio = round(precio_lista * rng.uniform(0.9, 1.1), 2)
        cantidad = rng.randint(1, 4)
        devolucion = DEVOLUCION[categoria]
        estado = rng.choices(["ENTREGADO", "CANCELADO", "DEVUELTO"],
                             weights=[90 - devolucion, 10, devolucion])[0]
        pago = rng.choices(list(PAGOS), weights=PAGOS.values())[0]
        # Errores típicos del sistema de origen
        r = rng.random()
        if r < 0.10:
            ciudad = ciudad.upper()
        elif r < 0.20:
            ciudad = f" {ciudad.lower()}  "
        if rng.random() < 0.05:
            estado = estado.lower()
        if rng.random() < 0.02:
            cantidad = ""
        if rng.random() < 0.01:
            precio = -precio
        filas.append([id_pedido, fecha, ciudad, categoria, producto, cantidad, precio, pago, estado])
    duplicados = rng.sample(dia_anterior, len(dia_anterior) // 50)  # pedidos reenviados
    dia_anterior = filas
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, lineterminator="\n")
        w.writerow(["id_pedido", "fecha", "ciudad", "categoria", "producto",
                    "cantidad", "precio_unitario", "metodo_pago", "estado"])
        w.writerows(filas + duplicados)

%cd {TF}
!ls datos/raw | head -3
print("...")

/content/trabajo_final
pedidos_2026-10-01.csv
pedidos_2026-10-02.csv
pedidos_2026-10-03.csv
...


## Parte 1 — Revisar los archivos (Línea de comandos · 3.5 pts)
Antes de procesar nada, un *data engineer* revisa qué llegó. Usa **solo comandos de terminal** (`!comando` para una línea, o `%%bash` al inicio de la celda para varias). Ya estás ubicado en `/content/trabajo_final`.

| Comando | ¿Qué hace? |
|---|---|
| `ls -lh carpeta` | Lista archivos con su tamaño |
| `ls carpeta \| wc -l` | Cuenta cuántos archivos hay |
| `du -sh carpeta` | Tamaño total de una carpeta |
| `head -n 5 archivo` | Muestra las primeras líneas |
| `find carpeta -type f -empty` | Busca archivos vacíos |
| `mv origen destino` | Mueve (o renombra) archivos |
| `wc -l archivos` | Cuenta líneas |
| `grep -ci texto` | Cuenta líneas que contienen un texto (sin importar mayúsculas) |

### ✏️ Ejercicio 1.1 — Explorar la carpeta raw (1 pts)
1. Lista los archivos de `datos/raw` con su tamaño.
2. Cuenta cuántos archivos hay y el tamaño total de la carpeta.
3. Muestra las 5 primeras líneas del archivo del 1 de octubre.
4. Responde: ¿qué columnas tiene cada archivo? ¿Ves algún archivo raro en el listado?

In [4]:
%%bash
# 1. Archivos con tamaño
ls -lh datos/raw


total 3.8M
-rw-r--r-- 1 root root 106K Nov  7 22:20 pedidos_2026-10-01.csv
-rw-r--r-- 1 root root 138K Nov  7 22:20 pedidos_2026-10-02.csv
-rw-r--r-- 1 root root 103K Nov  7 22:20 pedidos_2026-10-03.csv
-rw-r--r-- 1 root root 127K Nov  7 22:20 pedidos_2026-10-04.csv
-rw-r--r-- 1 root root 165K Nov  7 22:20 pedidos_2026-10-05.csv
-rw-r--r-- 1 root root 113K Nov  7 22:20 pedidos_2026-10-06.csv
-rw-r--r-- 1 root root 140K Nov  7 22:20 pedidos_2026-10-07.csv
-rw-r--r-- 1 root root 166K Nov  7 22:20 pedidos_2026-10-08.csv
-rw-r--r-- 1 root root 112K Nov  7 22:20 pedidos_2026-10-09.csv
-rw-r--r-- 1 root root 115K Nov  7 22:20 pedidos_2026-10-10.csv
-rw-r--r-- 1 root root 135K Nov  7 22:20 pedidos_2026-10-11.csv
-rw-r--r-- 1 root root    0 Nov  7 22:20 pedidos_2026-10-12.csv
-rw-r--r-- 1 root root 109K Nov  7 22:20 pedidos_2026-10-13.csv
-rw-r--r-- 1 root root 135K Nov  7 22:20 pedidos_2026-10-14.csv
-rw-r--r-- 1 root root 117K Nov  7 22:20 pedidos_2026-10-15.csv
-rw-r--r-- 1 root root 137K N

In [5]:
%%bash
# 2. Conteo y tamaño total
ls -1 datos/raw | wc -l
du -sh datos/raw

31
3.8M	datos/raw


In [6]:
%%bash
# 3. Primeras 5 líneas del archivo del 1 de octubre
head -n 5 datos/raw/*10-01*


id_pedido,fecha,ciudad,categoria,producto,cantidad,precio_unitario,metodo_pago,estado
1,2026-10-01,Lima,Moda,Casaca,1,226.48,Transferencia,DEVUELTO
2,2026-10-01, cusco  ,Hogar,Licuadora,4,263.65,Yape,ENTREGADO
3,2026-10-01,Chiclayo,Moda,Zapatillas,3,263.77,Transferencia,ENTREGADO
4,2026-10-01,Chiclayo,Tecnologia,Celular,3,1511.42,Transferencia,ENTREGADO


In [7]:
%%bash
# 4. Columnas de cada archivo (si son CSV)
for f in datos/raw/*.csv; do echo "== $f"; head -n 1 "$f"; done

== datos/raw/pedidos_2026-10-01.csv
id_pedido,fecha,ciudad,categoria,producto,cantidad,precio_unitario,metodo_pago,estado
== datos/raw/pedidos_2026-10-02.csv
id_pedido,fecha,ciudad,categoria,producto,cantidad,precio_unitario,metodo_pago,estado
== datos/raw/pedidos_2026-10-03.csv
id_pedido,fecha,ciudad,categoria,producto,cantidad,precio_unitario,metodo_pago,estado
== datos/raw/pedidos_2026-10-04.csv
id_pedido,fecha,ciudad,categoria,producto,cantidad,precio_unitario,metodo_pago,estado
== datos/raw/pedidos_2026-10-05.csv
id_pedido,fecha,ciudad,categoria,producto,cantidad,precio_unitario,metodo_pago,estado
== datos/raw/pedidos_2026-10-06.csv
id_pedido,fecha,ciudad,categoria,producto,cantidad,precio_unitario,metodo_pago,estado
== datos/raw/pedidos_2026-10-07.csv
id_pedido,fecha,ciudad,categoria,producto,cantidad,precio_unitario,metodo_pago,estado
== datos/raw/pedidos_2026-10-08.csv
id_pedido,fecha,ciudad,categoria,producto,cantidad,precio_unitario,metodo_pago,estado
== datos/raw/pedidos_202

# ¿Qué columnas tiene cada archivo?
 28 de los 30 archivos tienen exactamente las mismas 9 columnas, en el mismo orden:<br><br>

<strong>id_pedido, fecha, ciudad, categoria, producto, cantidad, precio_unitario, metodo_pago, estado
</strong><br>

<br>Del 02 al 11, del 13 al 24 y del 26 al 31 de octubre el esquema es consistente, sin columnas extra, faltantes ni renombradas.

# ¿Ves algún archivo raro en el listado?
Sí, son los siguientes casos:
1. <strong>pedidos_2026-10-12.csv y pedidos_2026-10-25.csv</strong>: No muestran ni siquiera la fila de encabezado. Lo más probable es que estén vacíos (0 bytes) o que su primera línea esté en blanco.
2. <strong>Falta el archivo del 1 de octubre</strong>: El listado arranca en 2026-10-02, así que hay un hueco al inicio del mes.

### ✏️ Ejercicio 1.2 — Separar los archivos vacíos (1.5 pts)
1. Encuentra los archivos vacíos de `datos/raw` con `find`.
2. Muévelos a `datos/cuarentena` (con `mv` o con `find ... -exec mv {} destino \;`).
3. Comprueba: lista `datos/cuarentena` y cuenta cuántos archivos quedan en `datos/raw`.
4. Responde: ¿por qué conviene apartarlos en lugar de borrarlos?

In [8]:
%%bash
# Encuentra los archivos vacíos
find datos/raw -type f -empty

datos/raw/pedidos_2026-10-25.csv
datos/raw/pedidos_2026-10-12.csv


In [9]:
%%bash
# Archivos movidos a cuarentena
mkdir -p datos/cuarentena
find datos/raw -type f -empty -exec mv -v {} datos/cuarentena/ \;

renamed 'datos/raw/pedidos_2026-10-25.csv' -> 'datos/cuarentena/pedidos_2026-10-25.csv'
renamed 'datos/raw/pedidos_2026-10-12.csv' -> 'datos/cuarentena/pedidos_2026-10-12.csv'


In [10]:
%%bash
# Comprueba: lista datos/cuarentena y cuenta cuántos archivos quedan
ls -l datos/cuarentena
ls -1 datos/raw | wc -l

total 0
-rw-r--r-- 1 root root 0 Nov  7 22:20 pedidos_2026-10-12.csv
-rw-r--r-- 1 root root 0 Nov  7 22:20 pedidos_2026-10-25.csv
29


4. ¿Por qué apartarlos en lugar de borrarlos?

Es irreversible, al borrar  no tiene forma de deshacer. Si resulta que el archivo vacío era un error tuyo o del proceso y no de la fuente, moverlo te deja recuperarlo.

Es evidencia de un problema. Un archivo vacío suele ser síntoma de una falla en la extracción o la carga. Si lo borras, pierdes la pista de que ese día falló, y nadie vuelve a pedir esos datos.


### ✏️ Ejercicio 1.3 — Contar registros antes de Spark (1 pts)
1. Cuenta las líneas de todos los CSV de `datos/raw` juntos con `wc -l` (la última línea muestra el total).
2. Calcula cuántas **filas de datos** hay: total de líneas menos una cabecera por archivo. Anota el número: lo compararás en el Ejercicio 2.1.
3. Con `grep`, cuenta cuántas líneas contienen la palabra `cancelado` sin importar mayúsculas.

In [11]:
%%bash
# Cuenta las líneas de todos los CSV
wc -l datos/raw/*.csv

   1622 datos/raw/pedidos_2026-10-01.csv
   2096 datos/raw/pedidos_2026-10-02.csv
   1557 datos/raw/pedidos_2026-10-03.csv
   1936 datos/raw/pedidos_2026-10-04.csv
   2512 datos/raw/pedidos_2026-10-05.csv
   1698 datos/raw/pedidos_2026-10-06.csv
   2090 datos/raw/pedidos_2026-10-07.csv
   2492 datos/raw/pedidos_2026-10-08.csv
   1681 datos/raw/pedidos_2026-10-09.csv
   1714 datos/raw/pedidos_2026-10-10.csv
   2016 datos/raw/pedidos_2026-10-11.csv
   1640 datos/raw/pedidos_2026-10-13.csv
   2012 datos/raw/pedidos_2026-10-14.csv
   1758 datos/raw/pedidos_2026-10-15.csv
   2058 datos/raw/pedidos_2026-10-16.csv
   1970 datos/raw/pedidos_2026-10-17.csv
   2048 datos/raw/pedidos_2026-10-18.csv
   1832 datos/raw/pedidos_2026-10-19.csv
   2264 datos/raw/pedidos_2026-10-20.csv
   2144 datos/raw/pedidos_2026-10-21.csv
   1824 datos/raw/pedidos_2026-10-22.csv
   2312 datos/raw/pedidos_2026-10-23.csv
   2171 datos/raw/pedidos_2026-10-24.csv
   1716 datos/raw/pedidos_2026-10-26.csv
   2520 datos/ra

In [12]:
%%bash
# Calcula cuántas filas de datos
echo $(( $(cat datos/raw/*.csv | wc -l) - $(ls datos/raw/*.csv | wc -l) ))

57303


In [13]:
%%bash
# Cuenta cuántas líneas contienen la palabra cancelado
cat datos/raw/*.csv | grep -i cancelado | wc -l

5616


## Parte 2 — Analizar con Spark (DataFrames · 12 pts)
Cada ejercicio puedes resolverlo con la **API de DataFrames** o con **Spark SQL**; el Ejercicio 2.7 es obligatoriamente con SQL. **No uses RDDs.**

| API de DataFrames | Spark SQL | ¿Qué hace? |
|---|---|---|
| `df.select("a", "b")` | `SELECT a, b` | Elegir columnas |
| `df.filter(F.col("a") > 0)` | `WHERE a > 0` | Filtrar filas |
| `df.dropDuplicates(["id"])` | `SELECT DISTINCT ...` | Quitar duplicados |
| `df.dropna(subset=["a"])` | `WHERE a IS NOT NULL` | Quitar nulos |
| `df.withColumn("t", expr)` | `SELECT ..., expr AS t` | Crear o reemplazar una columna |
| `df.groupBy("a").agg(F.sum("b"))` | `GROUP BY a` | Agrupar y agregar |
| `df.groupBy("a").count()` | `COUNT(*)` | Contar por grupo |
| `df.orderBy(F.desc("b"))` | `ORDER BY b DESC` | Ordenar |
| `df.createOrReplaceTempView("v")` | `spark.sql("SELECT ... FROM v")` | Usar SQL sobre un DataFrame |

> 💡 Para que las sumas grandes no salgan en notación científica: `.cast("decimal(18,2)")` o `CAST(SUM(x) AS DECIMAL(18,2))`.

### ✏️ Ejercicio 2.1 — Leer los pedidos (1 pts)
1. Lee **todos** los CSV de `datos/raw` en un solo DataFrame llamado `pedidos` (pista: `spark.read.csv(f"{RAW}/*.csv", header=True, inferSchema=True)`).
2. Muestra el esquema con `printSchema()` y cuenta las filas.
3. Responde: ¿coincide con el número de filas de datos que calculaste en el Ejercicio 1.3?

In [14]:
#Leer Archvio
import os, glob
RAW = "/content/trabajo_final/datos/raw"
pedidos = spark.read.csv(f"{RAW}/*.csv", header=True, inferSchema=True)
#Carga Esquema e informacion Archvios
print(f"#Leer Archvio")
pedidos.printSchema()
print(f"Total de pedidos:{pedidos.count()}")
print("¿coincide con el número de filas de datos que calculaste en el Ejercicio 1.3?")
if 57332 == pedidos.count():
  print("Coincide")
else:
  print(f"No Coincide el valor del ejeracio 1.3 es de {577332} y el proceso presento {pedidos.count()}")

#Leer Archvio
root
 |-- id_pedido: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- estado: string (nullable = true)

Total de pedidos:57303
¿coincide con el número de filas de datos que calculaste en el Ejercicio 1.3?
No Coincide el valor del ejeracio 1.3 es de 577332 y el proceso presento 57303


### ✏️ Ejercicio 2.2 — Explorar la calidad de los datos (1.5 pts)
1. Muestra 5 filas solo de las columnas `fecha`, `ciudad`, `producto`, `cantidad` y `estado`.
2. Muestra los valores distintos de `ciudad` y cuéntalos. ¿Por qué hay más de 6?
3. Cuenta cuántas filas tienen `cantidad` nula y cuántas tienen `precio_unitario` menor o igual a 0.
4. Cuenta cuántos `id_pedido` están repetidos (filas totales menos `id_pedido` distintos).

In [15]:
#1 Muestra 5 filas .
pedidos.select("fecha", "ciudad", "producto", "cantidad", "estado").show(5, truncate=False)
#2 Muestra los valores distintos de ciudad y cuéntalos.
from pyspark.sql import functions as F
ciudades = pedidos.groupBy("ciudad").count().orderBy("ciudad")
ciudades.show(100, truncate=False)
print("Ciudades distintas:", pedidos.select("ciudad").distinct().count())
#3 Cuenta cuántas filas tienen cantidad nula y cuántas tienen precio_unitario menor o igual a 0.
nulos_cantidad = pedidos.filter(F.col("cantidad").isNull()).count()
precio_invalido = pedidos.filter(F.col("precio_unitario") <= 0).count()

print("cantidad nula:", nulos_cantidad)
print("precio_unitario <= 0:", precio_invalido)
#4 Cuenta cuántos id_pedido están repetidos (filas totales menos id_pedido distintos).
total = pedidos.count()
distintos = pedidos.select("id_pedido").distinct().count()
print("Repetidos:", total - distintos)

+----------+--------+----------+--------+---------+
|fecha     |ciudad  |producto  |cantidad|estado   |
+----------+--------+----------+--------+---------+
|2026-10-27|Lima    |Casaca    |2       |ENTREGADO|
|2026-10-27|Lima    |Zapatillas|1       |CANCELADO|
|2026-10-27|Trujillo|Cafetera  |4       |ENTREGADO|
|2026-10-27|CUSCO   |Audifonos |4       |ENTREGADO|
|2026-10-27|Arequipa|Audifonos |4       |ENTREGADO|
+----------+--------+----------+--------+---------+
only showing top 5 rows

+-----------+-----+
|ciudad     |count|
+-----------+-----+
| arequipa  |856  |
| chiclayo  |549  |
| cusco     |731  |
| lima      |2260 |
| piura     |566  |
| trujillo  |686  |
|AREQUIPA   |848  |
|Arequipa   |6805 |
|CHICLAYO   |576  |
|CUSCO      |728  |
|Chiclayo   |4629 |
|Cusco      |6006 |
|LIMA       |2296 |
|Lima       |18429|
|PIURA      |558  |
|Piura      |4614 |
|TRUJILLO   |694  |
|Trujillo   |5472 |
+-----------+-----+

Ciudades distintas: 18
cantidad nula: 1127
precio_unitario <= 0: 5

### ✏️ Ejercicio 2.3 — Limpiar los pedidos (2 pts)
Crea el DataFrame `df_limpio` a partir de `pedidos`:
1. Elimina los duplicados por `id_pedido`.
2. Elimina las filas con `cantidad` nula y con `precio_unitario <= 0`.
3. Normaliza `ciudad` (quita espacios y deja la primera letra en mayúscula: `F.initcap(F.trim(...))`).
4. Normaliza `estado` en mayúsculas y sin espacios (`F.upper(F.trim(...))`).
5. Agrega la columna `total = cantidad * precio_unitario` redondeada a 2 decimales.

Muestra cuántas filas había antes y después, y los valores distintos de `ciudad` (deben ser 6) y de `estado` (deben ser 3).

In [16]:
antes = pedidos.count()
df_limpio = (
    pedidos
    # 1. Duplicados por id_pedido
    .dropDuplicates(["id_pedido"])
    # 2. Cantidad nula y precio inválido
    .filter(F.col("cantidad").isNotNull())
    .filter(F.col("precio_unitario") > 0)
    # 3. Normalizar ciudad
    .withColumn("ciudad", F.initcap(F.trim(F.col("ciudad"))))
    # 4. Normalizar estado
    .withColumn("estado", F.upper(F.trim(F.col("estado"))))
    # 5. Total redondeado a 2 decimales
    .withColumn("total", F.round(F.col("cantidad") * F.col("precio_unitario"), 2))
)

despues = df_limpio.count()
print(f"Filas antes: {antes}")
print(f"Filas después: {despues}")
print(f"Filas eliminadas: {antes - despues}")

df_limpio.select("ciudad").distinct().orderBy("ciudad").show(truncate=False)
print("Ciudades distintas:", df_limpio.select("ciudad").distinct().count())

df_limpio.select("estado").distinct().orderBy("estado").show(truncate=False)
print("Estados distintos:", df_limpio.select("estado").distinct().count())

Filas antes: 57303
Filas después: 54566
Filas eliminadas: 2737
+--------+
|ciudad  |
+--------+
|Arequipa|
|Chiclayo|
|Cusco   |
|Lima    |
|Piura   |
|Trujillo|
+--------+

Ciudades distintas: 6
+---------+
|estado   |
+---------+
|CANCELADO|
|DEVUELTO |
|ENTREGADO|
+---------+

Estados distintos: 3


### ✏️ Ejercicio 2.4 — Filtrar pedidos (1.5 pts)
1. Crea el DataFrame `entregados` solo con los pedidos en estado `ENTREGADO` y cuéntalos.
2. ¿Cuántos pedidos entregados en **Lima**, pagados con **Yape**, tienen un `total` mayor a **1000**?
3. Muestra los 5 pedidos entregados de mayor `total` (columnas `id_pedido`, `ciudad`, `producto`, `cantidad`, `total`).

In [17]:
#1. Pedidos en estado ENTREGADO
entregados = df_limpio.filter(F.col("estado") == "ENTREGADO")
print("Pedidos entregados:", entregados.count())
#2. Cuántos pedidos entregados
conteo = (entregados
          .filter(F.col("ciudad") == "Lima")
          .filter(F.lower(F.trim(F.col("metodo_pago"))) == "yape")
          .filter(F.col("total") > 1000)
          .count())

print("Entregados en Lima con Yape y total > 1000:", conteo)
#3. Muestra los 5 pedidos entregados de mayor
(entregados
    .select("id_pedido", "ciudad", "producto", "cantidad", "total")
    .orderBy(F.desc("total"))
    .show(5, truncate=False))

Pedidos entregados: 45665
Entregados en Lima con Yape y total > 1000: 1812
+---------+--------+--------+--------+--------+
|id_pedido|ciudad  |producto|cantidad|total   |
+---------+--------+--------+--------+--------+
|52149    |Chiclayo|Laptop  |4       |14076.4 |
|47253    |Chiclayo|Laptop  |4       |14071.4 |
|12957    |Lima    |Laptop  |4       |14070.04|
|45636    |Arequipa|Laptop  |4       |14065.44|
|55798    |Piura   |Laptop  |4       |14064.44|
+---------+--------+--------+--------+--------+
only showing top 5 rows



### ✏️ Ejercicio 2.5 — Conteos (1.5 pts)
1. Cuenta los pedidos por `estado`, de mayor a menor.
2. Cuenta los pedidos por `metodo_pago`, de mayor a menor.
3. Responde: ¿qué porcentaje de los pedidos se cancela? ¿Cuál es el método de pago más usado?

In [18]:
#1.Cuenta los pedidos por estado
por_estado = (df_limpio.groupBy("estado")
              .count()
              .orderBy(F.desc("count")))
por_estado.show(truncate=False)
#2.Cuenta los pedidos por metodo_pago
por_pago = (df_limpio.groupBy("metodo_pago")
            .count()
            .orderBy(F.desc("count")))
por_pago.show(truncate=False)
#3.Responde
total = df_limpio.count()
cancelados = df_limpio.filter(F.col("estado") == "CANCELADO").count()

print(f"Cancelados: {cancelados} de {total} = {cancelados / total * 100:.2f}%")

top_pago = por_pago.first()
print("Método más usado:", top_pago["metodo_pago"], "con", top_pago["count"], "pedidos")

+---------+-----+
|estado   |count|
+---------+-----+
|ENTREGADO|45665|
|CANCELADO|5339 |
|DEVUELTO |3562 |
+---------+-----+

+-------------+-----+
|metodo_pago  |count|
+-------------+-----+
|Tarjeta      |21958|
|Yape         |16287|
|Efectivo     |8211 |
|Transferencia|8110 |
+-------------+-----+

Cancelados: 5339 de 54566 = 9.78%
Método más usado: Tarjeta con 21958 pedidos


### ✏️ Ejercicio 2.6 — Ventas por ciudad (2 pts)
Con los pedidos **entregados**, crea el DataFrame `ventas_por_ciudad` con una fila por ciudad y estas columnas:
- `pedidos`: número de pedidos
- `unidades`: suma de `cantidad`
- `ventas`: suma de `total` (en decimal, sin notación científica)
- `ticket_promedio`: promedio de `total` redondeado a 2 decimales

Ordénalo de mayor a menor `ventas` y muéstralo. Responde: ¿qué ciudad vende más y cuál menos?

In [19]:
ventas_por_ciudad = (
    entregados
    .groupBy("ciudad")
    .agg(
        F.count("*").alias("pedidos"),
        F.sum("cantidad").alias("unidades"),
        F.sum("total").cast("decimal(18,2)").alias("ventas"),
        F.round(F.avg("total"), 2).alias("ticket_promedio"),
    )
    .orderBy(F.desc("ventas"))
)

ventas_por_ciudad.show(truncate=False)

# Respuesta: ciudad que más y que menos vende
mas = ventas_por_ciudad.first()
menos = ventas_por_ciudad.orderBy(F.asc("ventas")).first()
print(f"Ciudad que más vende:   {mas['ciudad']} ({mas['ventas']})")
print(f"Ciudad que menos vende: {menos['ciudad']} ({menos['ventas']})")

+--------+-------+--------+-----------+---------------+
|ciudad  |pedidos|unidades|ventas     |ticket_promedio|
+--------+-------+--------+-----------+---------------+
|Lima    |18341  |45793   |31123284.87|1696.92        |
|Arequipa|6760   |16824   |11215911.73|1659.16        |
|Cusco   |5961   |14913   |9804943.29 |1644.85        |
|Trujillo|5483   |13648   |8951107.45 |1632.52        |
|Piura   |4554   |11480   |7852829.32 |1724.38        |
|Chiclayo|4566   |11282   |7766903.83 |1701.03        |
+--------+-------+--------+-----------+---------------+

Ciudad que más vende:   Lima (31123284.87)
Ciudad que menos vende: Chiclayo (7766903.83)


### ✏️ Ejercicio 2.7 — Las mismas preguntas con Spark SQL (1.5 pts)
**Este ejercicio es obligatorio con SQL.**
1. Registra `df_limpio` como la vista temporal `pedidos_limpios`.
2. Con SQL, obtén por `categoria` el número de pedidos **entregados** y la suma de sus ventas, de mayor a menor.
3. Con SQL, obtén por `categoria` el número de pedidos **devueltos**, de mayor a menor. Responde: ¿qué categoría tiene más devoluciones?

In [20]:
#1  vista temporal
df_limpio.createOrReplaceTempView("pedidos_limpios")
#2. pedidos entregados
spark.sql("""
    SELECT
        categoria,
        COUNT(*)                          AS pedidos_entregados,
        CAST(SUM(total) AS DECIMAL(18,2)) AS ventas
    FROM pedidos_limpios
    WHERE estado = 'ENTREGADO'
    GROUP BY categoria
    ORDER BY ventas DESC
""").show(truncate=False)
#3. pedidos devueltos
devoluciones = spark.sql("""
    SELECT
        categoria,
        COUNT(*) AS pedidos_devueltos
    FROM pedidos_limpios
    WHERE estado = 'DEVUELTO'
    GROUP BY categoria
    ORDER BY pedidos_devueltos DESC
""")
devoluciones.show(truncate=False)

+----------+------------------+-----------+
|categoria |pedidos_entregados|ventas     |
+----------+------------------+-----------+
|Tecnologia|20216             |59937691.17|
|Moda      |15713             |10007489.55|
|Hogar     |9736              |6769799.77 |
+----------+------------------+-----------+

+----------+-----------------+
|categoria |pedidos_devueltos|
+----------+-----------------+
|Moda      |2340             |
|Tecnologia|873              |
|Hogar     |349              |
+----------+-----------------+



¿qué categoría tiene más devoluciones? Moda

### ✏️ Ejercicio 2.8 — Guardar el resultado (1 pts)
1. Guarda `ventas_por_ciudad` como CSV con cabecera en `resultados/ventas_por_ciudad_spark`, en **un solo archivo** (pista: `coalesce(1)` y `mode("overwrite")`).
2. Lista la carpeta que creó Spark. Responde: ¿por qué Spark crea una **carpeta** y no un archivo?

In [21]:
(ventas_por_ciudad
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv("/content/trabajo_final/resultados//ventas_por_ciudad_spark"))
print(os.listdir("/content/trabajo_final/resultados/ventas_por_ciudad_spark"))
origen = glob.glob("/content/trabajo_final/resultados//ventas_por_ciudad_spark/part-*.csv")[0]
shutil.copy(origen, "/content/trabajo_final/resultados//ventas_por_ciudad.csv")

['.part-00000-695f46f5-842b-4af3-9ce0-b2d7c90f5ed1-c000.csv.crc', 'part-00000-695f46f5-842b-4af3-9ce0-b2d7c90f5ed1-c000.csv', '._SUCCESS.crc', '_SUCCESS']


'/content/trabajo_final/resultados//ventas_por_ciudad.csv'

¿por qué Spark crea una carpeta y no un archivo? Porque Spark escribe en paralelo. Cada partición del DataFrame la guarda una tarea distinta, y cada tarea produce su propio archivo

## Parte 3 — Cerrar el proceso (Línea de comandos · 2 pts)
Spark ya terminó. Ahora deja las carpetas ordenadas para el próximo mes: lo que está en `raw/` ya se procesó y el resultado debe tener un nombre claro.

### ✏️ Ejercicio 3.1 — Mover los CSV procesados (1 pts)
1. Mueve todos los CSV de `datos/raw` a `datos/procesados` con `mv`.
2. Comprueba que `datos/raw` quedó vacía y cuenta cuántos archivos hay en `datos/procesados`.
3. Responde: ¿por qué es importante sacar los archivos de `raw/` después de procesarlos?

In [22]:
%%bash
#1 Mueve todos los CSV
mkdir -p datos/procesados
mv -v datos/raw/*.csv datos/procesados/
#2 Comprueba que datos/raw quedó vacía
ls -A datos/raw | wc -l
ls -1 datos/procesados | wc -l


renamed 'datos/raw/pedidos_2026-10-01.csv' -> 'datos/procesados/pedidos_2026-10-01.csv'
renamed 'datos/raw/pedidos_2026-10-02.csv' -> 'datos/procesados/pedidos_2026-10-02.csv'
renamed 'datos/raw/pedidos_2026-10-03.csv' -> 'datos/procesados/pedidos_2026-10-03.csv'
renamed 'datos/raw/pedidos_2026-10-04.csv' -> 'datos/procesados/pedidos_2026-10-04.csv'
renamed 'datos/raw/pedidos_2026-10-05.csv' -> 'datos/procesados/pedidos_2026-10-05.csv'
renamed 'datos/raw/pedidos_2026-10-06.csv' -> 'datos/procesados/pedidos_2026-10-06.csv'
renamed 'datos/raw/pedidos_2026-10-07.csv' -> 'datos/procesados/pedidos_2026-10-07.csv'
renamed 'datos/raw/pedidos_2026-10-08.csv' -> 'datos/procesados/pedidos_2026-10-08.csv'
renamed 'datos/raw/pedidos_2026-10-09.csv' -> 'datos/procesados/pedidos_2026-10-09.csv'
renamed 'datos/raw/pedidos_2026-10-10.csv' -> 'datos/procesados/pedidos_2026-10-10.csv'
renamed 'datos/raw/pedidos_2026-10-11.csv' -> 'datos/procesados/pedidos_2026-10-11.csv'
renamed 'datos/raw/pedidos_2026-

¿por qué es importante sacar los archivos de raw/ después de procesarlos?

- Evita procesarlos dos veces. Si raw/ funciona como bandeja de entrada, todo lo que hay dentro se considera pendiente. Dejarlos ya procesados haría que la siguiente ejecución duplique pedidos y distorsione ventas y conteos.

### ✏️ Ejercicio 3.2 — Dejar el resultado con un nombre claro (1 pts)
1. Mueve el archivo `part-*.csv` que creó Spark a `resultados/ventas_por_ciudad.csv` (con `mv` también se renombra).
2. Borra la carpeta `resultados/ventas_por_ciudad_spark`.
3. Muestra el contenido de `resultados/ventas_por_ciudad.csv` y la estructura final con `find datos resultados -type f | sort`.

In [23]:
%%bash
#1 Mueve el archivo part-*.csv
mv -v resultados/ventas_por_ciudad_spark/part-*.csv resultados/ventas_por_ciudad.csv
#2 Borra la carpeta resultados/ventas_por_ciudad_spark
rm -r resultados/ventas_por_ciudad_spark
#3 Muestra el contenido de resultados/ventas_por_ciudad.csv
cat resultados/ventas_por_ciudad.csv
find datos resultados -type f | sort

renamed 'resultados/ventas_por_ciudad_spark/part-00000-695f46f5-842b-4af3-9ce0-b2d7c90f5ed1-c000.csv' -> 'resultados/ventas_por_ciudad.csv'
ciudad,pedidos,unidades,ventas,ticket_promedio
Lima,18341,45793,31123284.87,1696.92
Arequipa,6760,16824,11215911.73,1659.16
Cusco,5961,14913,9804943.29,1644.85
Trujillo,5483,13648,8951107.45,1632.52
Piura,4554,11480,7852829.32,1724.38
Chiclayo,4566,11282,7766903.83,1701.03
datos/cuarentena/pedidos_2026-10-12.csv
datos/cuarentena/pedidos_2026-10-25.csv
datos/procesados/pedidos_2026-10-01.csv
datos/procesados/pedidos_2026-10-02.csv
datos/procesados/pedidos_2026-10-03.csv
datos/procesados/pedidos_2026-10-04.csv
datos/procesados/pedidos_2026-10-05.csv
datos/procesados/pedidos_2026-10-06.csv
datos/procesados/pedidos_2026-10-07.csv
datos/procesados/pedidos_2026-10-08.csv
datos/procesados/pedidos_2026-10-09.csv
datos/procesados/pedidos_2026-10-10.csv
datos/procesados/pedidos_2026-10-11.csv
datos/procesados/pedidos_2026-10-13.csv
datos/procesados/pedidos_2

## Parte 4 — Entregar con Git (2.5 pts)
Esta parte se hace **en la terminal de tu computadora** (Git Bash en Windows, Terminal en Mac/Linux), siguiendo los pasos del documento del trabajo final.

1. Descarga el resultado ejecutando la celda de abajo (se guarda en tu carpeta de Descargas).
2. Escribe tu nombre y el link de tu repositorio al inicio de este notebook.
3. **Entorno de ejecución → Reiniciar y ejecutar todo**, y luego **Archivo → Descargar → Descargar .ipynb**.
4. Sube los dos archivos a tu repositorio público con `git add`, `git commit` y `git push`.

In [24]:
from google.colab import files
files.download(f"{RESULTADOS}/ventas_por_ciudad.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>